# Swiggy Exploratory Data Analysis (EDA)

This notebook answers all **10 Core Business Questions** using the cleaned Swiggy dataset (`swiggy_cleaned.csv`).

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('swiggy_cleaned.csv')
print(f"Loaded cleaned dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(df.head(5))


Loaded cleaned dataset: 139,321 rows, 13 columns
   restaurant_name  ...                                         offer_name
0  La Pino'Z Pizza  ...  FLAT DEAL | FLAT ₹125 OFF | USE FLAT125ABOVE ₹...
1  The Second Wife  ...  30% OFF UPTO ₹75 | USE TRYNEWABOVE ₹149, FLAT ...
2      Tasty Bites  ...              FLAT ₹120 OFF | USE AXIS120ABOVE ₹500
3      Food Studio  ...  50% OFF UPTO ₹100 | USE TRYNEWABOVE ₹129, FLAT...
4     Roll Express  ...  DEAL OF DAY | 10% OFF UPTO ₹40 | USE STEALDEAL...

[5 rows x 13 columns]


### Q1: Which locations have the most restaurants?

In [2]:
loc_counts = df['location'].value_counts()
top_15_locations = pd.DataFrame({
    'Restaurant Count': loc_counts.head(15),
    '% Share': (loc_counts.head(15) / len(df) * 100).round(2),
    'Cumulative %': (loc_counts.head(15).cumsum() / len(df) * 100).round(2)
})
print(f"Total unique locations: {df['location'].nunique()}")
print(top_15_locations)


Total unique locations: 581
                Restaurant Count  % Share  Cumulative %
location                                               
Kanpur                      2000     1.44          1.44
Central-Goa                 1998     1.43          2.87
Hyderabad                   1998     1.43          4.30
Indore                      1998     1.43          5.74
Nagpur                      1998     1.43          7.17
Vadodara                    1998     1.43          8.61
Lucknow                     1996     1.43         10.04
Jaipur                      1990     1.43         11.47
Kanchrapara                 1977     1.42         12.89
Kolkata                     1974     1.42         14.30
Bhubaneswar                 1906     1.37         15.67
Doddaballapura              1893     1.36         17.03
Mumbai                      1893     1.36         18.39
Coimbatore                  1889     1.36         19.74
Ahmedabad                   1801     1.29         21.04


### Q2: Which cuisines dominate?

In [3]:
print("--- Top 10 Primary Cuisines ---")
prim_df = pd.DataFrame({
    'Outlet Count': df['primary_cuisine'].value_counts().head(10),
    '% Share': (df['primary_cuisine'].value_counts().head(10) / len(df) * 100).round(2)
})
print(prim_df)

print("\n--- Top 10 All Cuisines Offered (Across Menus) ---")
all_cuisines = df['cuisine'].str.split(', ').explode()
all_cuisines_summary = pd.DataFrame({
    'Total Listings Offering': all_cuisines.value_counts().head(10),
    '% of All Restaurants': (all_cuisines.value_counts().head(10) / len(df) * 100).round(2)
})
print(all_cuisines_summary)


--- Top 10 Primary Cuisines ---
                 Outlet Count  % Share
primary_cuisine                       
North Indian            20154    14.47
Chinese                 18844    13.53
Indian                  16452    11.81
Biryani                  9798     7.03
Pizzas                   8602     6.17
South Indian             8190     5.88
Bakery                   7806     5.60
Beverages                7272     5.22
Snacks                   6287     4.51
Fast Food                5411     3.88

--- Top 10 All Cuisines Offered (Across Menus) ---
              Total Listings Offering  % of All Restaurants
cuisine                                                    
Chinese                         37527                 26.94
North Indian                    28722                 20.62
Indian                          23160                 16.62
Snacks                          17339                 12.45
Biryani                         14942                 10.72
Fast Food                   

### Q3: What is the typical restaurant rating?

In [4]:
r = df['rating'].dropna()
print("Rating Summary Statistics:")
print(f"Mean: {r.mean():.2f} | Median: {r.median():.2f} | Mode: {r.mode()[0]:.2f} | Std: {r.std():.2f}")
print(f"IQR (25% - 75%): {r.quantile(0.25):.2f} to {r.quantile(0.75):.2f}")

rating_bins = pd.cut(r, bins=[0, 3.0, 3.5, 4.0, 4.5, 5.0], labels=['< 3.0', '3.0 - 3.4', '3.5 - 3.9', '4.0 - 4.4', '4.5 - 5.0'])
rating_dist = pd.DataFrame({
    'Count': rating_bins.value_counts().sort_index(),
    '% Share': (rating_bins.value_counts(normalize=True).sort_index() * 100).round(2)
})
print("\nRating Tier Distribution:")
print(rating_dist)


Rating Summary Statistics:
Mean: 4.04 | Median: 4.10 | Mode: 4.30 | Std: 0.50
IQR (25% - 75%): 3.80 to 4.30

Rating Tier Distribution:
           Count  % Share
rating                   
< 3.0       5004     4.73
3.0 - 3.4   9978     9.43
3.5 - 3.9  29352    27.73
4.0 - 4.4  51156    48.32
4.5 - 5.0  10374     9.80


### Q4: What is the typical price?

In [5]:
p = df['cost_for_two']
print("Cost for Two Statistics (INR):")
print(f"Mean: Rs. {p.mean():.2f} | Median: Rs. {p.median():.2f} | Mode: Rs. {p.mode()[0]:.2f}")
print(f"IQR: Rs. {p.quantile(0.25):.2f} to Rs. {p.quantile(0.75):.2f}")

price_tiers = pd.cut(p, bins=[0, 150, 250, 500, 1000, 10000], labels=['Budget (<= 150)', 'Affordable (151 - 250)', 'Mid-Range (251 - 500)', 'Premium (501 - 1000)', 'Luxury (> 1000)'])
tier_df = pd.DataFrame({
    'Count': price_tiers.value_counts().sort_index(),
    '% Share': (price_tiers.value_counts(normalize=True).sort_index() * 100).round(2)
})
print("\nPrice Tiers Breakdown:")
print(tier_df)


Cost for Two Statistics (INR):
Mean: Rs. 268.91 | Median: Rs. 250.00 | Mode: Rs. 200.00
IQR: Rs. 200.00 to Rs. 300.00

Price Tiers Breakdown:
                        Count  % Share
cost_for_two                          
Budget (<= 150)         21450    15.40
Affordable (151 - 250)  63321    45.45
Mid-Range (251 - 500)   49825    35.76
Premium (501 - 1000)     4280     3.07
Luxury (> 1000)           445     0.32


### Q5: Does price relate to rating?

In [6]:
clean_pr = df.dropna(subset=['rating', 'cost_for_two']).copy()
pearson = clean_pr['cost_for_two'].corr(clean_pr['rating'])
spearman = clean_pr['cost_for_two'].rank().corr(clean_pr['rating'].rank())
print(f"Pearson Correlation: {pearson:.4f} | Spearman Rank: {spearman:.4f}")

clean_pr['price_tier'] = pd.cut(clean_pr['cost_for_two'], bins=[0, 150, 250, 500, 1000, 10000], labels=['Budget (<= 150)', 'Affordable (151 - 250)', 'Mid-Range (251 - 500)', 'Premium (501 - 1000)', 'Luxury (> 1000)'])
tier_rating = clean_pr.groupby('price_tier', observed=False)['rating'].agg([
    'count', 'mean', 'median', lambda x: round((x >= 4.0).mean() * 100, 2)
]).rename(columns={'<lambda_0>': '% Rated >= 4.0'}).round(2)
print("\nAverage Rating by Price Tier:")
print(tier_rating)


Pearson Correlation: 0.0634 | Spearman Rank: 0.0589

Average Rating by Price Tier:
                        count  mean  median  % Rated >= 4.0
price_tier                                                 
Budget (<= 150)         16244  4.03     4.1           63.93
Affordable (151 - 250)  47589  4.01     4.1           63.50
Mid-Range (251 - 500)   38386  4.06     4.2           69.53
Premium (501 - 1000)     3312  4.16     4.2           78.93
Luxury (> 1000)           333  4.18     4.3           77.48


### Q6: Does rating relate to popularity?

In [7]:
pop_df = df.dropna(subset=['rating', 'rating_count']).copy()
print(f"Correlation (Rating vs Reviews): {pop_df['rating'].corr(pop_df['rating_count']):.4f}")

pop_df['popularity_tier'] = pd.cut(pop_df['rating_count'], bins=[0, 9, 49, 99, 499, 999, 4999, 100000], labels=['Micro (2-9)', 'Niche (10-49)', 'Moderate (50-99)', 'High (100-499)', 'Very High (500-999)', 'Massive (1K-4.9K)', 'Superstar (5K+)'])
pop_tier_df = pop_df.groupby('popularity_tier', observed=False)['rating'].agg([
    'count', 'mean', 'median', lambda x: round((x >= 4.0).mean() * 100, 2)
]).rename(columns={'<lambda_0>': '% Rated >= 4.0'}).round(2)
print("\nPerformance Across Popularity Tiers:")
print(pop_tier_df)


Correlation (Rating vs Reviews): 0.1131

Performance Across Popularity Tiers:
                     count  mean  median  % Rated >= 4.0
popularity_tier                                         
Micro (2-9)          18168  3.91     4.0           52.16
Niche (10-49)        26799  3.95     4.0           55.46
Moderate (50-99)     10902  4.03     4.1           64.51
High (100-499)       23137  4.08     4.1           71.36
Very High (500-999)   8307  4.14     4.2           78.10
Massive (1K-4.9K)    13337  4.18     4.2           83.46
Superstar (5K+)       5214  4.23     4.3           89.43


### Q7: How does pure veg compare with other restaurants?

In [8]:
veg_comparison = df.groupby('is_pure_veg').agg(
    total_count=('restaurant_name', 'count'),
    pct_share=('restaurant_name', lambda x: round(len(x) / len(df) * 100, 2)),
    avg_price=('cost_for_two', 'mean'),
    median_price=('cost_for_two', 'median'),
    avg_rating=('rating', 'mean'),
    median_rating=('rating', 'median'),
    avg_reviews=('rating_count', 'mean'),
    avg_offers=('offer_count', 'mean')
).round(2)
print("Pure Veg vs Non-Veg Comparison:")
print(veg_comparison.T)


Pure Veg vs Non-Veg Comparison:
is_pure_veg       False     True 
total_count    80826.00  58495.00
pct_share         58.01     41.99
avg_price        289.82    240.03
median_price     250.00    200.00
avg_rating         4.03      4.06
median_rating      4.10      4.20
avg_reviews      596.61    478.56
avg_offers         3.26      3.08


### Q8: How are promotional offers distributed?

In [9]:
offer_counts = df['offer_count'].value_counts().sort_index()
offer_df = pd.DataFrame({
    'Restaurants': offer_counts,
    '% Share': (offer_counts / len(df) * 100).round(2)
})
print("Offer Distribution:")
print(offer_df)
print(f"\nRestaurants with >= 1 offer: {(df['offer_count'] > 0).sum():,} ({(df['offer_count'] > 0).mean()*100:.2f}%)")


Offer Distribution:
             Restaurants  % Share
offer_count                      
0                   1752     1.26
1                  34606    24.84
2                  14882    10.68
3                   8759     6.29
4                  43622    31.31
5                  35700    25.62

Restaurants with >= 1 offer: 137,569 (98.74%)


### Q9: Which restaurants are highly rated and well-reviewed?

In [10]:
gold_tier = df[(df['rating'] >= 4.5) & (df['rating_count'] >= 1000)]
print(f"Restaurants with Rating >= 4.5 and Reviews >= 1,000: {len(gold_tier):,} ({len(gold_tier)/len(df)*100:.2f}%)")

print("\nTop 10 Outlets (Rating >= 4.8, Sorted by Review Volume):")
top_10_outlets = gold_tier.sort_values(by=['rating', 'rating_count'], ascending=[False, False])[
    ['restaurant_name', 'location', 'primary_cuisine', 'rating', 'rating_count', 'cost_for_two']
].head(10)
print(top_10_outlets.to_string())


Restaurants with Rating >= 4.5 and Reviews >= 1,000: 2,230 (1.60%)

Top 10 Outlets (Rating >= 4.8, Sorted by Review Volume):
                                restaurant_name    location primary_cuisine  rating  rating_count  cost_for_two
9592                          Om Veg Restaurant  Aurangabad   Maharashtrian     5.0        1000.0           150
19655                         Agra Chat Bhandar    Bhilwara       Fast Food     5.0        1000.0           100
85137                          Apni Nighi Rasoi    Ludhiana          Indian     5.0        1000.0           200
15780          Amrit'S Bikaner Mishthan Bhandar     Barnala        Desserts     4.9        1000.0           200
57710                        Royal Pizza Canter      Indore          Pizzas     4.9        1000.0           250
53997                         Natural Ice Cream       Hosur       Ice Cream     4.8       10000.0           150
61399                         Natural Ice Cream      Jaipur       Ice Cream     4.8       1

### Q10: Which locations look strongest across multiple metrics?

In [11]:
city_stats = df.groupby('location').agg(
    total_restaurants=('restaurant_name', 'count'),
    avg_rating=('rating', 'mean'),
    pct_high_rated=('rating', lambda x: (x >= 4.2).mean() * 100),
    avg_reviews=('rating_count', 'mean'),
    median_cost=('cost_for_two', 'median'),
    avg_offers=('offer_count', 'mean')
)

q_cities = city_stats[city_stats['total_restaurants'] >= 500].copy()

def norm(s):
    return (s - s.min()) / (s.max() - s.min())

q_cities['composite_score'] = (
    norm(q_cities['avg_rating']) * 0.35 +
    norm(q_cities['avg_reviews']) * 0.30 +
    norm(q_cities['total_restaurants']) * 0.20 +
    norm(q_cities['avg_offers']) * 0.15
) * 100

top_cities_df = q_cities.sort_values('composite_score', ascending=False).head(10).round(2)[
    ['total_restaurants', 'avg_rating', 'avg_reviews', 'median_cost', 'composite_score']
]
print("Top 10 Strongest Cities (Composite Score):")
print(top_cities_df)


Top 10 Strongest Cities (Composite Score):
                    total_restaurants  avg_rating  ...  median_cost  composite_score
location                                           ...                              
Coimbatore                       1889        4.18  ...        200.0            82.08
Thiruvananthapuram                837        4.24  ...        250.0            81.56
Bhubaneswar                      1906        4.12  ...        248.0            75.64
Surat                            1751        4.20  ...        300.0            74.06
Bangalore                        1720        4.13  ...        264.5            73.62
Kochi                            1470        4.26  ...        249.0            72.48
Vadodara                         1998        4.15  ...        250.0            72.33
Guwahati                         1683        4.12  ...        300.0            72.31
Delhi                            1783        3.99  ...        250.0            70.66
Nagpur                